# 02 — Classical Baselines (UBCF)

Phase 2 (user-based):
- MiniBatch K-Means user clustering
- UBCF user-kNN with adjusted cosine / mean-centered weighted regression
- Elbow + Silhouette tuning

Outputs: `user_clusters.npy`, `ubcf_metrics.csv`

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score

IN_DIR = Path("data/processed_32m_ubcf")
OUT_DIR = Path("data/processed_32m_ubcf")

SEED = 42
K_LIST = [16, 24, 32, 40, 48, 64]
TOPK_NEIGHBORS = 40
TOPN = 10
EVAL_USERS = 500

user_sparse = load_npz(IN_DIR / "user_sparse.npz").tocsr().astype(np.float32)
user_latent = np.load(IN_DIR / "user_latent.npy").astype(np.float32)

print(user_sparse.shape, user_latent.shape)

(200948, 84432) (200948, 32)


In [5]:
# Tuning: Elbow + Silhouette
rows = []
for k in K_LIST:
    km = MiniBatchKMeans(n_clusters=k, batch_size=2048, random_state=SEED, n_init=10, max_iter=200)
    labels = km.fit_predict(user_latent)
    sil = silhouette_score(user_latent, labels, sample_size=min(10000, len(user_latent)), random_state=SEED) if len(np.unique(labels)) > 1 else np.nan
    rows.append({"k": k, "inertia": float(km.inertia_), "silhouette": float(sil)})

tuning_df = pd.DataFrame(rows)
best_k = int(tuning_df.loc[tuning_df["silhouette"].idxmax(), "k"]) if tuning_df["silhouette"].notna().any() else int(tuning_df.loc[tuning_df["inertia"].idxmin(), "k"])
print(tuning_df)
print(f"best_k={best_k}")

    k     inertia  silhouette
0  16  60213620.0    0.122855
1  24  55720808.0    0.103695
2  32  52145616.0    0.101817
3  40  49758800.0    0.090975
4  48  48141120.0    0.071086
5  64  46180988.0    0.061759
best_k=16


In [6]:
# Fit final user clusters
kmeans = MiniBatchKMeans(n_clusters=best_k, batch_size=2048, random_state=SEED, n_init=10, max_iter=200)
user_clusters = kmeans.fit_predict(user_latent).astype(np.int32)

# Build adjusted-cosine (mean-centered cosine) user-user similarity on small active subset for speed
rng = np.random.default_rng(SEED)
user_nnz = np.diff(user_sparse.indptr)
active = np.where(user_nnz >= 20)[0]
eval_users = rng.choice(active, size=min(EVAL_USERS, len(active)), replace=False)

all_metrics = []
hits, ndcgs, precisions = [], [], []
preds, trues = [], []

global_mean = float(user_sparse.data.mean()) if user_sparse.nnz else 3.5

for u in eval_users:
    start, end = user_sparse.indptr[u], user_sparse.indptr[u+1]
    user_items = user_sparse.indices[start:end]
    user_vals = user_sparse.data[start:end]
    if len(user_items) < 3:
        continue

    held_ix = int(rng.integers(len(user_items)))
    held_item = int(user_items[held_ix])
    held_true = float(user_vals[held_ix])

    train_items = set(user_items.tolist())
    train_items.discard(held_item)

    neigh_pool = np.where(user_clusters == user_clusters[u])[0]
    neigh_pool = neigh_pool[neigh_pool != u]
    if len(neigh_pool) == 0:
        continue

    u_row = user_sparse[u]
    u_mean = float(u_row.data.mean()) if u_row.nnz else global_mean
    sims = []
    for v in neigh_pool:
        v_row = user_sparse[v]
        if v_row.nnz == 0:
            continue
        common = np.intersect1d(u_row.indices, v_row.indices, assume_unique=False)
        if len(common) < 2:
            continue
        u_map = {i:r for i, r in zip(u_row.indices, u_row.data)}
        v_map = {i:r for i, r in zip(v_row.indices, v_row.data)}
        u_vec = np.array([u_map[i] for i in common], dtype=np.float32)
        v_vec = np.array([v_map[i] for i in common], dtype=np.float32)
        u_c = u_vec - u_vec.mean()
        v_c = v_vec - v_vec.mean()
        den = float(np.linalg.norm(u_c) * np.linalg.norm(v_c) + 1e-8)
        s = float(np.dot(u_c, v_c) / den)
        sims.append((v, s))

    if not sims:
        continue

    sims.sort(key=lambda x: abs(x[1]), reverse=True)
    sims = sims[:TOPK_NEIGHBORS]

    # Weighted regression prediction for held item
    num, den = 0.0, 0.0
    for v, s in sims:
        v_row = user_sparse[v]
        v_map = {i:r for i, r in zip(v_row.indices, v_row.data)}
        if held_item in v_map:
            v_mean = float(v_row.data.mean()) if v_row.nnz else global_mean
            num += s * (float(v_map[held_item]) - v_mean)
            den += abs(s)

    pred = u_mean + (num / (den + 1e-8)) if den > 0 else u_mean
    pred = float(np.clip(pred, 0.5, 5.0))

    preds.append(pred)
    trues.append(held_true)

    # Ranking over candidate movies
    candidates = np.setdiff1d(np.arange(user_sparse.shape[1]), np.array(list(train_items), dtype=np.int32), assume_unique=False)
    if len(candidates) > 2000:
        candidates = rng.choice(candidates, size=2000, replace=False)

    scores = []
    for m in candidates:
        num2, den2 = 0.0, 0.0
        for v, s in sims:
            v_row = user_sparse[v]
            v_map = {i:r for i, r in zip(v_row.indices, v_row.data)}
            if int(m) in v_map:
                v_mean = float(v_row.data.mean()) if v_row.nnz else global_mean
                num2 += s * (float(v_map[int(m)]) - v_mean)
                den2 += abs(s)
        sc = u_mean + (num2 / (den2 + 1e-8)) if den2 > 0 else u_mean
        scores.append(float(sc))

    scores = np.array(scores, dtype=np.float32)
    top_idx = np.argsort(-scores)[:TOPN]
    top_items = candidates[top_idx]
    hit = 1.0 if held_item in top_items else 0.0
    hits.append(hit)
    precisions.append(hit / TOPN)
    if hit:
        rank = int(np.where(top_items == held_item)[0][0]) + 1
        ndcgs.append(float(1.0 / np.log2(rank + 1)))
    else:
        ndcgs.append(0.0)

rmse = float(np.sqrt(mean_squared_error(trues, preds))) if preds else np.nan
mae = float(mean_absolute_error(trues, preds)) if preds else np.nan
metrics_df = pd.DataFrame([{"model": "ubcf_user_knn_adjusted_cosine", "rmse": rmse, "mae": mae, "hr@10": float(np.mean(hits)) if hits else np.nan, "ndcg@10": float(np.mean(ndcgs)) if ndcgs else np.nan, "precision@10": float(np.mean(precisions)) if precisions else np.nan, "k_clusters": best_k}])

np.save(OUT_DIR / "user_clusters.npy", user_clusters)
metrics_df.to_csv(OUT_DIR / "ubcf_metrics.csv", index=False)
tuning_df.to_csv(OUT_DIR / "ubcf_tuning.csv", index=False)

print(metrics_df.to_string(index=False))

                        model     rmse      mae  hr@10  ndcg@10  precision@10  k_clusters
ubcf_user_knn_adjusted_cosine 0.829645 0.608413  0.004 0.001528        0.0004          16
